# MarketLens: Data Cleaning & Exploratory Data Analysis (EDA)

This notebook demonstrates the Batch 2 data cleaning, exploratory analysis, and core analytics layer on the customer feedback dataset.

**Pipeline Flow:** `Raw CSV -> Ingestion -> Cleaning Pipeline -> EDA & Core Analytics -> Export`

In [1]:
from pathlib import Path

from marketlens.analytics import (
    aggregate_by,
    dataset_overview,
    filter_dataset,
    product_service_analysis,
    rating_analysis,
    temporal_analysis,
    text_analysis,
)
from marketlens.cleaning import (
    CleaningPipeline,
    run_pipeline,
)
from marketlens.ingestion import ingest

## 1. Data Ingestion
Run CSV ingestion with validation, column mapping, and record classification.

In [2]:
csv_path = Path("../data/raw/synthetic_feedback_demo.csv")
if not csv_path.exists():
    csv_path = Path("data/raw/synthetic_feedback_demo.csv")

ingest_result = ingest(csv_path, rating_scale="1-5")
print(ingest_result.summary())
print("\nRow Accounting:", ingest_result.row_accounting)

## 2. Data Cleaning Pipeline
Apply text normalisation, date standardisation, rating clamping, and duplicate detection without data loss.

In [3]:
pipeline = CleaningPipeline(ingest_result)
cleaned_df, cleaning_report = pipeline.run(include_flagged=False, rating_scale="1-5")

print(
    f"Retained Rows: "
    f"{cleaning_report['rows_retained']} / {cleaning_report['input_row_count']}"
)
print("Operations Applied:", cleaning_report["cleaning_operations"])
cleaned_df.head()

## 3. Dataset Overview
Overview metrics, category breakdowns, date span, and data quality summary.

In [4]:
overview = dataset_overview(cleaned_df)
print("Total Records:", overview["total_records"])
print("Products:", overview["product_breakdown"])
print("Sources:", overview["source_breakdown"])
print("Date Coverage:", overview["date_coverage"])
print("Quality Summary:", overview["quality_summary"])

## 4. Rating Analysis
Rating distribution, percentages, central tendencies, and trends.

In [5]:
ratings = rating_analysis(cleaned_df, rating_scale="1-5")
print(
    f"Mean: {ratings['mean']} | Median: {ratings['median']} | "
    f"Range: [{ratings['min']}, {ratings['max']}]"
)
print("Rating Distribution:", ratings["distribution"])
print("Rating Percentages (%):", ratings["percentages"])

## 5. Temporal Analysis & Period-over-Period Trends

In [6]:
temporal = temporal_analysis(cleaned_df, period="M")
print("Volume by Month:", temporal["volume_by_period"])
print("Mean Rating by Month:", temporal["rating_trend_by_period"])
print("Period-over-Period Changes:", temporal["period_over_period_changes"])
if temporal["period_warnings"]:
    print("Sufficiency Warnings:", temporal["period_warnings"])

## 6. Text Analysis (Unigrams & Bigrams)
Extract meaningful terms with stopword removal. Does not perform sentiment classification.

In [7]:
text_stats = text_analysis(cleaned_df, min_frequency=2)
print("Total Words:", text_stats["total_words"])
print("Top Unigrams:", text_stats["unigrams"])
print("Top Bigrams:", text_stats["bigrams"])
print("Disclaimer:", text_stats["disclaimer"])

## 7. Product & Channel Analysis

In [8]:
prod_stats = product_service_analysis(cleaned_df)
print("Volume by Product:", prod_stats["volume_by_product"])
print("Avg Rating by Product:", prod_stats["avg_rating_by_product"])
print("Volume by Source:", prod_stats["volume_by_source"])
print("Avg Rating by Source:", prod_stats["avg_rating_by_source"])

## 8. Composable Filtering & Aggregations

In [9]:
filtered_df = filter_dataset(
    cleaned_df,
    product="Laptop",
    min_rating=4.0,
    include_flagged=False,
)
print(f"Filtered Laptop Reviews (rating >= 4.0): {len(filtered_df)}")

agg = aggregate_by(
    cleaned_df, group_by="product_or_service", value="rating", agg_func="mean"
)
display(agg) if "display" in locals() else print(agg)

## 9. End-to-End Execution & Persistence

In [10]:
summary = run_pipeline(
    csv_path,
    include_flagged=False,
    rating_scale="1-5",
    output_dir="../data/processed" if Path("../data").exists() else "data/processed",
    report_dir="../reports" if Path("../reports").exists() else "reports",
)
print("Cleaned CSV Saved:", summary["csv_path"])
print("Cleaning Report Saved:", summary["cleaning_report_path"])
print("Analytics Report Saved:", summary["analytics_report_path"])